# QRURLs - Interactive QR Code & URL Shortener

This notebook provides an interactive UI to generate QR codes and shorten URLs.

## Setup

First, install the package with notebook support:
```bash
pip install qrurls[notebook]
# or with uv:
uv pip install qrurls[notebook]
```

In [ ]:
# Import required libraries
import ipywidgets as widgets
from IPython.display import display, Image, HTML, clear_output
from qrurls import QRURLs, QRGenerator, URLShortener
import io
from datetime import datetime

## Interactive UI

Use the widgets below to generate QR codes and shorten URLs!

In [ ]:
# Create UI widgets
url_input = widgets.Text(
    value='https://www.example.com',
    placeholder='Enter URL here',
    description='URL:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='80%')
)

service_dropdown = widgets.Dropdown(
    options=['tinyurl', 'isgd', 'vgd'],
    value='tinyurl',
    description='Service:',
    style={'description_width': '100px'}
)

box_size_slider = widgets.IntSlider(
    value=10,
    min=5,
    max=20,
    step=1,
    description='QR Box Size:',
    style={'description_width': '100px'}
)

border_slider = widgets.IntSlider(
    value=4,
    min=1,
    max=10,
    step=1,
    description='QR Border:',
    style={'description_width': '100px'}
)

action_dropdown = widgets.Dropdown(
    options=['Both (Shorten + QR)', 'Shorten Only', 'QR Only'],
    value='Both (Shorten + QR)',
    description='Action:',
    style={'description_width': '100px'}
)

generate_button = widgets.Button(
    description='Generate',
    button_style='success',
    icon='check'
)

output_area = widgets.Output()

# Function to handle button click
def on_generate_click(b):
    with output_area:
        clear_output(wait=True)
        
        url = url_input.value.strip()
        
        if not url:
            print("⚠️ Please enter a URL")
            return
        
        if not url.startswith(('http://', 'https://')):
            print("⚠️ URL must start with http:// or https://")
            return
        
        try:
            print(f"🔄 Processing URL: {url}\n")
            
            qrurls = QRURLs(
                service=service_dropdown.value,
                box_size=box_size_slider.value,
                border=border_slider.value
            )
            
            action = action_dropdown.value
            
            if action == 'Both (Shorten + QR)':
                # Shorten and generate QR
                short_url, qr_img = qrurls.process(url)
                
                print(f"✅ Original URL: {url}")
                print(f"✅ Shortened URL: {short_url}\n")
                
                # Save and display QR code
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                filename = f'qr_{timestamp}.png'
                qr_img.save(filename)
                
                print(f"💾 QR code saved as: {filename}\n")
                print("📱 QR Code Preview:")
                display(Image(filename, width=300))
                
            elif action == 'Shorten Only':
                # Just shorten
                short_url = qrurls.shorten_only(url)
                print(f"✅ Original URL: {url}")
                print(f"✅ Shortened URL: {short_url}")
                
                # Create clickable link
                display(HTML(f'<a href="{short_url}" target="_blank">Click to test: {short_url}</a>'))
                
            elif action == 'QR Only':
                # Just generate QR
                qr_gen = QRGenerator(
                    box_size=box_size_slider.value,
                    border=border_slider.value
                )
                qr_img = qr_gen.generate(url)
                
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                filename = f'qr_{timestamp}.png'
                qr_img.save(filename)
                
                print(f"✅ QR code generated for: {url}")
                print(f"💾 Saved as: {filename}\n")
                print("📱 QR Code Preview:")
                display(Image(filename, width=300))
            
        except Exception as e:
            print(f"❌ Error: {str(e)}")

generate_button.on_click(on_generate_click)

# Display UI
display(widgets.VBox([
    widgets.HTML(value="<h3>🔗 QRURLs Generator</h3>"),
    url_input,
    action_dropdown,
    service_dropdown,
    box_size_slider,
    border_slider,
    generate_button,
    output_area
]))

## Advanced Usage

You can also use the library directly in code cells:

In [ ]:
# Example: Batch processing multiple URLs
urls_to_process = [
    'https://github.com',
    'https://python.org',
    'https://jupyter.org'
]

qrurls = QRURLs(service='tinyurl')

print("Batch Processing URLs:\n")
for i, url in enumerate(urls_to_process, 1):
    try:
        short_url = qrurls.shorten_only(url)
        print(f"{i}. {url}")
        print(f"   → {short_url}\n")
    except Exception as e:
        print(f"{i}. {url}")
        print(f"   ❌ Error: {e}\n")

## Tips

- **Box Size**: Controls the size of each QR code square (5-20)
- **Border**: Controls the white border around the QR code (1-10)
- **Services**: 
  - `tinyurl` - Simple and reliable
  - `isgd` - HTTPS support
  - `vgd` - Alternative to is.gd

All QR codes are automatically saved with timestamps in the current directory.